<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Solution: One-Class SVM and the PyOD toolbox

**[One-Class SVM and the PyOD toolbox](https://learning.nextia-ai.com/courses/anomaly-detection/m04/one-class-svm-and-the-pyod-toolbox/)** · Anomaly Detection: Find What Is Unusual · Module 4, lesson 4 of 5 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** draw a boundary around normal data with a One-Class SVM, run four detectors with one interface in PyOD, and combine their scores, and you will see when a combination helps and when it hurts.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 7 MB of data, and PyOD with its helper numba). No account. The notebook takes about a minute to run. |
| **You should know** | kNN distance, LOF and Isolation Forest, from the first three lessons of this module, for example [Isolation Forest](https://learning.nextia-ai.com/courses/anomaly-detection/m04/isolation-forest/). |
| **You do not need** | A local project. The setup below downloads everything. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m04/one-class-svm-and-the-pyod-toolbox/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M04-L04-one-class-svm-and-the-pyod-toolbox/one-class-svm-and-pyod-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2, pyod 3.6.7. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pyod==3.6.7
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib', 'pyod']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`payments.csv.gz`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py` and `features.py`, the course's shared code. It needs an internet connection and no account. It takes about 10 seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "payments.csv.gz": "c4e18c1bbaf422cfc99821f2a1b65cf0e8274f835c40ca757a37a531eeccd542",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py", "features.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A fence around normal: a tiny example

A farmer puts a fence around the field where the sheep usually graze. A sheep outside the fence is worth a look. The farmer did not need to see any lost sheep to build the fence: only the normal field.

A **One-Class SVM** does the same with data. It learns from normal points only, and draws a smooth boundary around them. A new point outside the boundary is unusual. The setting **nu** (the Greek letter, said "new") is the share of training points that may stay outside the fence: a small nu gives a wide fence, a large nu a tight one.

Run the next cell. It builds 60 normal points and 2 strange ones, and fits a One-Class SVM with three values of nu.

> **Check.** You should see 7 of 60 normal points outside with nu = 0.1, and 17 with nu = 0.3. Both strange points are outside every time.

In [ ]:
from sklearn.svm import OneClassSVM
rng = np.random.default_rng(3)
normal = rng.normal([5.0, 5.0], [1.0, 0.7], size=(60, 2)).round(2)   # 60 normal points around (5, 5)
strange = np.array([[9.0, 8.5], [1.2, 8.0]])                         # 2 strange points
for nu in [0.02, 0.1, 0.3]:
    svm = OneClassSVM(kernel="rbf", nu=nu, gamma=0.3).fit(normal)    # learns from the normal points only
    print(f"nu = {nu}: {(svm.predict(normal) == -1).sum()} of 60 normal points outside, "
          f"{(svm.predict(strange) == -1).sum()} of 2 strange points outside")

## 2. Four detectors, one interface

So far, every detector had its own code: `NearestNeighbors`, `LocalOutlierFactor`, `IsolationForest`. **PyOD** (Python Outlier Detection) is a library that puts more than 40 detectors behind the same two calls: `fit(X)` to learn normal, and `decision_function(X)` to score new points, where higher always means more unusual.

Run the next cell. It loads the payments, builds the behaviour features and scales them. It also draws the sample of 20,000 reference payments that the slow detectors use (the same sample as in the earlier lessons).

In [ ]:
import compare, features
from sklearn.preprocessing import StandardScaler
from pyod.models.knn import KNN
from pyod.models.lof import LOF
from pyod.models.iforest import IForest
from pyod.models.ocsvm import OCSVM
payments = pd.read_csv("data/payments.csv.gz", parse_dates=["ts"])
reference, evaluation = compare.split(payments)
behaviour = features.behaviour(payments)
scaler = StandardScaler().fit(behaviour.loc[reference.index])
X_ref, X_ev = scaler.transform(behaviour.loc[reference.index]), scaler.transform(behaviour.loc[evaluation.index])
sample = np.random.default_rng(0).choice(len(X_ref), 20_000, replace=False)   # for the slow detectors
print(X_ref.shape, X_ev.shape)

Run the next cell. It fits the four detectors of this module in one loop, with the settings of the comparison table, and scores the evaluation months. It takes about 20 seconds.

> **Check.** You should see kNN distance 0.505, LOF 0.082, Isolation Forest 0.500 and One-Class SVM 0.480.

In [ ]:
detectors = {
    "kNN distance":     (KNN(n_neighbors=10, method="largest"), X_ref[sample]),
    "LOF":              (LOF(n_neighbors=35, novelty=True), X_ref[sample]),
    "Isolation Forest": (IForest(n_estimators=200, random_state=0, n_jobs=-1), X_ref),
    "One-Class SVM":    (OCSVM(kernel="rbf", nu=0.02, gamma="auto"), X_ref[sample[:10_000]]),
}
scores = {}
table = compare.Table()
for name, (model, train) in detectors.items():
    model.fit(train)                          # the same two calls for every detector
    scores[name] = model.decision_function(X_ev)   # higher = more unusual, for every PyOD detector
    table.add(compare.score_payments(evaluation, scores[name], name, "behaviour features", explain_minutes=3))
table.show()

## 3. Combine the scores

The four scores are on different scales: a distance, a ratio, a score near 0.5, an SVM value. To combine them, first turn each score into its **rank**: 0 for the most usual payment, 1 for the most unusual. Then take the average of the ranks. Run the next cell.

> **Predict.** Three detectors have about 0.5, one has 0.08. Will the average of all four be near 0.5?

> **Check.** You should see 0.211 for the average of 4 detectors.

In [ ]:
def average_rank(names):
    """Turn each score into its rank (0 = most usual, 1 = most unusual), then average the ranks."""
    return np.mean([pd.Series(scores[n]).rank(pct=True).to_numpy() for n in names], axis=0)

table.add(compare.score_payments(evaluation, average_rank(list(scores)), "average of 4 detectors",
                                 "behaviour features, mean rank", explain_minutes=4))
table.show()

The average fell to 0.211. LOF's ranks pull every card test down, so the combination misses most of them. A combination is only as good as the detectors that you put in it.

## 4. Is a combination steadier?

A combination is often called "steadier": it should have fewer bad days than one detector. Check it. The next cell computes the precision at the budget of each day for each detector, and summarises the 91 days.

In [ ]:
def daily_precision(score):
    alerts = compare.daily_alerts(evaluation, score)
    alerts["planted"] = alerts.payment_id.isin(compare.planted_payments().payment_id)
    return alerts[alerts.reviewed].groupby("day").planted.mean()

days = pd.DataFrame({name: daily_precision(s) for name, s in scores.items()})
print(days.describe().loc[["mean", "std", "min", "max"]].round(3))

Look at the `min` row: every detector has a day at 0. On some days, very few payments are planted, so 50 alerts cannot all be frauds. A combination does not change that. Its real value is that it does not depend on choosing the right single detector in advance.

## 5. Check yourself

**Your turn 1.** In the tiny example, how many of the 60 normal points are outside the boundary with nu = 0.3? Write it as `outside_03`.

In [ ]:
outside_03 = 17

In [ ]:
expect('outside_03', outside_03, 17)

**Your turn 2.** Average the ranks of three detectors only: kNN distance, Isolation Forest and One-Class SVM. Put its precision at the budget in `precision_3`. Use `average_rank()`.

In [ ]:
precision_3 = compare.score_payments(evaluation, average_rank(["kNN distance", "Isolation Forest", "One-Class SVM"]),
                                     "average of 3 detectors", "behaviour features, mean rank")["precision at budget"]
print(precision_3)

In [ ]:
expect('precision_3', precision_3, 0.504, tolerance=0.001)

**Your turn 3.** Which detector did you leave out, and why did that help? Write its name, as the table shows it, in `left_out`.

In [ ]:
left_out = "LOF"

In [ ]:
expect_hash('left_out', left_out, 'da28eefc5fe00f7b')

In the solution: 0.504 without LOF, against 0.211 with it, and between the best single detectors.

## Recap

- A **One-Class SVM** learns a boundary around normal data only; **nu** sets how tight it is (7 of 60 normal points outside at nu = 0.1, 17 at 0.3).
- **PyOD** runs many detectors with the same `fit` and `decision_function`, so trying a new one is one line.
- A **rank average** combines scores on different scales. With a bad member, it hurts (0.211); without it, it is close to the best (0.504).

The next lesson asks the question behind this whole module: did the better algorithm or the better features give the larger gain?

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Features decide more than the algorithm](https://learning.nextia-ai.com/courses/anomaly-detection/m04/features-decide-more-than-the-algorithm/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m04/one-class-svm-and-the-pyod-toolbox/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.